# Neurosurgery RAG on Google Colab

Local-only RAG pipeline over neurosurgery/glioblastoma research PDFs. No cloud calls — Ollama runs in Colab, embeddings and Q&A are local.

## Setup

1. **Enable GPU**: Runtime > Change runtime type > T4 GPU (required for indexing speed)
2. **Upload PDFs to Drive** once: Create `/MyDrive/neosurgery/pdfs/` folder, upload 1000 PDFs there
3. **Run cells in order**: 1 (Ollama) → 2 (code) → 3 (Drive mount) → 4 (patch embedder) → 5 (extract) → 6 (index) → 7 (save)
4. **Query**: Use cell 8 or the Resume cell to restore after disconnect

## Pipeline Stages

- **Extract** (cell 5): PDFs → markdown with YAML frontmatter (~5 min)
- **Index** (cell 6): Markdown → Chroma vector store with embeddings (~15-30 min on GPU)
- **Query** (cell 8): Interactive Q&A against the index

## Important Notes

- **PDFs not in repo** (3.4 GB). Store in Drive, symlink into Colab workspace.
- **Ollama must stay alive** for query cell to work. If Ollama restarts, re-run cell 1.
- **Save to Drive** after indexing (cell 7). Colab disk wipes on disconnect — rebuild is expensive.
- **Embedding model mismatch breaks queries**. Cell 4 patches both indexer and query module to use the same embedder.
- **Use `requirements-colab.txt`, not `requirements.txt`**. Pins that downgrade Colab's preinstalled stack (numpy, packaging, `requests`) break `langgraph`/`google-adk`/`opencv`. The Colab file is unpinned below, holds `requests==2.32.4` for `google-colab`, and caps OpenTelemetry at 1.42.1 for `google-adk`.
- **No `langchain-community`**. It requires `requests>=2.32.5`, which `google-colab` forbids. The code uses the split packages instead (`langchain-chroma`, `langchain-ollama`, `langchain-classic`).

In [ ]:
# 1. Ollama server (background) + verify
import subprocess, time, sys
import urllib.request

print("Installing Ollama...")
subprocess.run(["bash", "-c", "curl -fsSL https://ollama.com/install.sh | sh"], check=False)

print("Starting Ollama daemon...")
proc = subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(8)

# Wait for Ollama to be ready
for i in range(30):
    try:
        urllib.request.urlopen('http://localhost:11434/api/tags', timeout=1)
        print("✓ Ollama ready")
        break
    except:
        time.sleep(1)
else:
    print("✗ Ollama failed to start")
    sys.exit(1)

print("Pulling models...")
subprocess.run(["ollama", "pull", "mistral"], check=False)
subprocess.run(["ollama", "pull", "nomic-embed-text"], check=False)

In [ ]:
# 2. Code + deps
!git clone https://github.com/mhdramadhanarvin/glioblastoma-model.git /content/repo
%cd /content/repo
!pip install -q -r requirements-colab.txt

In [ ]:
# 3. PDFs from Drive (upload the pdfs/ folder there once, by hand)
from google.colab import drive
drive.mount('/content/drive')
!ln -sfn /content/drive/MyDrive/neosurgery/pdfs /content/repo/pdfs
!ls pdfs | wc -l

In [ ]:
# 4. OPTIONAL but recommended: embed with nomic-embed-text instead of mistral.
# mistral is a 7B chat model used as an embedder - ~100x slower for the same job.
# Indexer and query module must agree, so patch both.
import fileinput

def patch_file(path, old, new):
    with fileinput.FileInput(path, inplace=True) as f:
        for line in f:
            print(line.replace(old, new), end='')

print("Patching knowledge_indexer.py...")
patch_file('src/knowledge_indexer.py', 'model_name: str = "mistral"', 'model_name: str = "nomic-embed-text"')

print("Patching rag_query.py...")
patch_file('src/rag_query.py', 'model_name: str = "mistral"', 'model_name: str = "nomic-embed-text"')

print("Verifying patches...")
!grep -n 'model_name.*nomic' src/knowledge_indexer.py src/rag_query.py

In [ ]:
# 5. Extract PDFs to markdown
print("Starting PDF extraction...")
!python3 main.py extract --pdf-dir pdfs --output-dir data/markdown
print("✓ Extraction complete. Check data/markdown/ for .md files")

In [ ]:
# 6. Build vector index from markdown
import time
print("Starting indexing (this takes 10-30 min on GPU, hours on CPU)...")
start = time.time()
!python3 main.py index --markdown-dir data/markdown --index-dir data/vectorstore
elapsed = time.time() - start
print(f"✓ Indexing complete in {elapsed/60:.1f} minutes")

In [ ]:
# 7. Persist index to Drive (optional but recommended)
import shutil
from pathlib import Path

drive_path = Path('/content/drive/MyDrive/neosurgery')
drive_path.mkdir(parents=True, exist_ok=True)

print("Copying markdown and vectorstore to Drive...")
shutil.copytree('data/markdown', drive_path / 'markdown', dirs_exist_ok=True)
shutil.copytree('data/vectorstore', drive_path / 'vectorstore', dirs_exist_ok=True)
print(f"✓ Saved to {drive_path}")

In [ ]:
# 8. Query the knowledge base
from src.rag_query import MedicalRAG

# Load index from local copy or Drive if available
index_path = 'data/vectorstore'
if not Path(index_path).exists():
    drive_index = Path('/content/drive/MyDrive/neosurgery/vectorstore')
    if drive_index.exists():
        print("Loading index from Drive...")
        shutil.copytree(drive_index, index_path)

rag = MedicalRAG(index_path)
if not rag.load_index():
    print("Index not found. Run cells 5-6 first.")
else:
    rag.setup_qa_chain()
    
    # Example queries
    questions = [
        "What are the standard treatment options for glioblastoma?",
        "What is the prognosis for glioblastoma patients?",
        "Describe the role of surgery in glioblastoma management."
    ]
    
    for q in questions:
        print(f"\n{'='*60}")
        print(f"Q: {q}")
        print(f"{'='*60}")
        answer = rag.query(q)
        print(f"\nA: {answer}\n")

In [ ]:
# RESUME from saved index (after disconnect/reconnect)
# Run this if you already extracted and indexed in a prior session
from google.colab import drive
from pathlib import Path
import shutil

drive.mount('/content/drive')

# Copy saved data back to Colab disk
drive_data = Path('/content/drive/MyDrive/neosurgery')
if drive_data.exists():
    print(f"Found saved data in Drive at {drive_data}")
    if (drive_data / 'markdown').exists():
        shutil.copytree(drive_data / 'markdown', 'data/markdown', dirs_exist_ok=True)
        print(f"✓ Restored {len(list((Path('data/markdown').glob('*.md'))))} markdown files")
    if (drive_data / 'vectorstore').exists():
        shutil.copytree(drive_data / 'vectorstore', 'data/vectorstore', dirs_exist_ok=True)
        print("✓ Restored vectorstore index")
else:
    print("No saved data found in Drive. Run the full pipeline first.")